# Enhancement + Weak Recognition Pipeline

This notebook is a commented walkthrough of the full project pipeline used in the project.

The goal is to test whether lightweight image enhancement improves weak video-level recognition on smartphone-captured herbal plant videos.

The notebook is organized as a complete reproducible pipeline:

1. Set up paths and configuration.
2. Extract frames from raw videos.
3. Apply enhancement methods.
4. Generate before/after visual grids.
5. Extract handcrafted, Tiny CNN, and ResNet18 features.
6. Train recognition baselines.
7. Compare raw and enhanced results.

## 1. Imports and Configuration

These imports cover image processing, plotting, classical machine learning, and PyTorch models.

The pipeline uses fixed seeds so the train/test split and Tiny CNN training are reproducible.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Callable
import json
import os
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import ResNet18_Weights, resnet18

from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from IPython.display import Image, display

# Make results more reproducible across runs.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Resolve the repository root whether the notebook is opened from the root or notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_DIR = ROOT / "Dataset"
OUTPUT_DIR = ROOT / "outputs" / "project"
RAW_FRAMES_DIR = OUTPUT_DIR / "frames" / "raw"
ENHANCED_DIR = OUTPUT_DIR / "frames" / "enhanced"
GRID_DIR = OUTPUT_DIR / "grids"
REPORTS_DIR = OUTPUT_DIR / "reports"
CSV_PATH = OUTPUT_DIR / "frame_index.csv"
SUMMARY_PATH = REPORTS_DIR / "summary.json"
ACCURACY_TABLE_PATH = REPORTS_DIR / "accuracy_table.csv"
TORCH_CACHE_DIR = OUTPUT_DIR / "torch_cache"

# Keep downloaded pretrained weights inside the project output folder.
os.environ["TORCH_HOME"] = str(TORCH_CACHE_DIR)

ROOT, DATASET_DIR, OUTPUT_DIR

## 2. Helper Classes and Folders

`PipelineSpec` stores the name and function for each enhancement pipeline.

`TinyCNN` is intentionally small because the dataset is small and weakly labeled. It gives us a learned baseline, but it is not expected to beat pretrained features.

In [ ]:
@dataclass(frozen=True)
class PipelineSpec:
    """Named enhancement pipeline used to process every extracted frame."""

    name: str
    fn: Callable[[np.ndarray], np.ndarray]


class TinyCNN(nn.Module):
    """A small CNN trained from scratch on 96x96 frame crops."""

    def __init__(self, num_classes: int) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def ensure_dirs() -> None:
    """Create every output folder used by the notebook."""

    for path in [RAW_FRAMES_DIR, ENHANCED_DIR, GRID_DIR, REPORTS_DIR, TORCH_CACHE_DIR]:
        path.mkdir(parents=True, exist_ok=True)


def iter_videos() -> list[Path]:
    """Return all raw videos in a stable order."""

    return sorted(DATASET_DIR.glob("*.mp4"))


ensure_dirs()
print(f"Found {len(iter_videos())} videos")

## 3. Frame Extraction

Frames are sampled at approximately one-second intervals.

A cap of 30 frames per video prevents longer clips from dominating the dataset. Shorter videos naturally produce fewer than 30 frames.

Each extracted frame is indexed with:

- source video ID
- original video path
- frame index
- timestamp
- extracted frame path
- FPS and duration metadata

In [ ]:
def extract_frames(sample_seconds: float = 1.0, max_frames_per_video: int = 30) -> pd.DataFrame:
    """Extract sampled frames from every video and write a frame index CSV."""

    rows: list[dict[str, object]] = []

    for video_path in iter_videos():
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        total_seconds = frame_count / fps if fps > 0 else 0

        # Convert the requested time interval to a frame step.
        step = max(int(round(fps * sample_seconds)), 1) if fps > 0 else 1

        video_id = video_path.stem
        video_dir = RAW_FRAMES_DIR / video_id
        video_dir.mkdir(parents=True, exist_ok=True)

        saved = 0
        frame_idx = 0
        while cap.isOpened():
            ok, frame = cap.read()
            if not ok:
                break

            if frame_idx % step == 0:
                frame_name = f"{video_id}_f{frame_idx:05d}.jpg"
                frame_path = video_dir / frame_name
                cv2.imwrite(str(frame_path), frame)

                rows.append(
                    {
                        "video_id": video_id,
                        "video_file": str(video_path.relative_to(ROOT)),
                        "frame_idx": frame_idx,
                        "timestamp_sec": frame_idx / fps if fps > 0 else None,
                        "frame_file": str(frame_path.relative_to(ROOT)),
                        "fps": fps,
                        "frame_count": frame_count,
                        "duration_sec": total_seconds,
                    }
                )

                saved += 1
                if saved >= max_frames_per_video:
                    break

            frame_idx += 1

        cap.release()

    df = pd.DataFrame(rows).sort_values(["video_id", "frame_idx"]).reset_index(drop=True)
    df.to_csv(CSV_PATH, index=False)
    return df

In [ ]:
# Run this cell to extract frames. If outputs already exist, this refreshes the CSV and frame files.
frame_index = extract_frames()
print(f"Extracted {len(frame_index)} frames from {frame_index['video_id'].nunique()} videos")
frame_index.head()

## 4. Enhancement Functions

These are classical, lightweight enhancement methods.

They are intentionally explainable and conservative because the dataset is small and does not include clean target images for training a dedicated enhancement network.

In [ ]:
def white_balance_gray_world(image_rgb: np.ndarray) -> np.ndarray:
    """Apply simple gray-world white balance."""

    img = image_rgb.astype(np.float32)
    channel_means = img.reshape(-1, 3).mean(axis=0)
    overall_mean = channel_means.mean()
    gains = overall_mean / np.clip(channel_means, 1e-6, None)
    balanced = img * gains
    return np.clip(balanced, 0, 255).astype(np.uint8)


def gamma_correction(image_rgb: np.ndarray, gamma: float = 0.9) -> np.ndarray:
    """Apply nonlinear tone mapping through a lookup table."""

    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(image_rgb, table)


def clahe_luminance(image_rgb: np.ndarray, clip_limit: float = 2.0, grid_size: int = 8) -> np.ndarray:
    """Apply CLAHE only to the luminance channel to preserve color relationships."""

    lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(grid_size, grid_size))
    enhanced_l = clahe.apply(l_channel)
    merged = cv2.merge([enhanced_l, a_channel, b_channel])
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)


def denoise_and_sharpen(image_rgb: np.ndarray) -> np.ndarray:
    """Denoise slightly, then apply mild unsharp masking."""

    denoised = cv2.fastNlMeansDenoisingColored(image_rgb, None, 6, 6, 7, 21)
    blurred = cv2.GaussianBlur(denoised, (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(denoised, 1.35, blurred, -0.35, 0)
    return np.clip(sharpened, 0, 255).astype(np.uint8)

## 5. Enhancement Pipelines

Each pipeline receives one RGB image and returns one enhanced RGB image.

The tested variants are:

- CLAHE
- lighter CLAHE
- light gamma correction
- light gamma followed by light CLAHE
- mild sharpening

In [ ]:
def pipeline_clahe(image_rgb: np.ndarray) -> np.ndarray:
    return clahe_luminance(image_rgb, clip_limit=2.0, grid_size=8)


def pipeline_clahe_light(image_rgb: np.ndarray) -> np.ndarray:
    return clahe_luminance(image_rgb, clip_limit=1.2, grid_size=8)


def pipeline_gamma_light(image_rgb: np.ndarray) -> np.ndarray:
    return gamma_correction(image_rgb, gamma=1.08)


def pipeline_gamma_clahe_light(image_rgb: np.ndarray) -> np.ndarray:
    return clahe_luminance(gamma_correction(image_rgb, gamma=1.08), clip_limit=1.2, grid_size=8)


def pipeline_sharpen_light(image_rgb: np.ndarray) -> np.ndarray:
    blurred = cv2.GaussianBlur(image_rgb, (0, 0), sigmaX=0.8)
    return np.clip(cv2.addWeighted(image_rgb, 1.18, blurred, -0.18, 0), 0, 255).astype(np.uint8)


PIPELINES = [
    PipelineSpec("clahe", pipeline_clahe),
    PipelineSpec("clahe_light", pipeline_clahe_light),
    PipelineSpec("gamma_light", pipeline_gamma_light),
    PipelineSpec("gamma_clahe_light", pipeline_gamma_clahe_light),
    PipelineSpec("sharpen_light", pipeline_sharpen_light),
]

[p.name for p in PIPELINES]

In [ ]:
def run_pipelines(frame_index: pd.DataFrame) -> pd.DataFrame:
    """Apply every enhancement pipeline to every raw extracted frame."""

    rows = []

    for pipeline in PIPELINES:
        for _, row in frame_index.iterrows():
            image_bgr = cv2.imread(str(ROOT / row["frame_file"]))
            if image_bgr is None:
                continue

            image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
            enhanced = pipeline.fn(image_rgb)

            out_dir = ENHANCED_DIR / pipeline.name / row["video_id"]
            out_dir.mkdir(parents=True, exist_ok=True)
            out_path = out_dir / Path(row["frame_file"]).name
            cv2.imwrite(str(out_path), cv2.cvtColor(enhanced, cv2.COLOR_RGB2BGR))

            rows.append(
                {
                    "variant": pipeline.name,
                    "video_id": row["video_id"],
                    "frame_file": str(out_path.relative_to(ROOT)),
                }
            )

    return pd.DataFrame(rows)


enhanced_index = run_pipelines(frame_index)
print(f"Created {len(enhanced_index)} enhanced frames")
enhanced_index.head()

## 6. Before/After Grids

These grids are used for qualitative inspection.

They help show whether an enhancement makes leaves, stems, and contrast more visible before we look at recognition accuracy.

In [ ]:
def build_before_after_grids(frame_index: pd.DataFrame, max_examples: int = 6) -> list[str]:
    """Create one comparison grid for the first frame of several videos."""

    grid_paths: list[str] = []
    sample_df = frame_index.groupby("video_id", as_index=False).first().head(max_examples)

    for _, row in sample_df.iterrows():
        raw_bgr = cv2.imread(str(ROOT / row["frame_file"]))
        if raw_bgr is None:
            continue

        raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
        fig, axes = plt.subplots(1, 1 + len(PIPELINES), figsize=(16, 4))
        axes[0].imshow(raw_rgb)
        axes[0].set_title("raw")
        axes[0].axis("off")

        for i, pipeline in enumerate(PIPELINES, start=1):
            enhanced_path = ENHANCED_DIR / pipeline.name / row["video_id"] / Path(row["frame_file"]).name
            enhanced_bgr = cv2.imread(str(enhanced_path))
            enhanced_rgb = cv2.cvtColor(enhanced_bgr, cv2.COLOR_BGR2RGB)
            axes[i].imshow(enhanced_rgb)
            axes[i].set_title(pipeline.name)
            axes[i].axis("off")

        fig.suptitle(row["video_id"])
        fig.tight_layout()
        out_path = GRID_DIR / f"{row['video_id']}_grid.png"
        fig.savefig(out_path, dpi=160)
        plt.close(fig)
        grid_paths.append(str(out_path.relative_to(ROOT)))

    return grid_paths


grids = build_before_after_grids(frame_index)
for grid in grids[:2]:
    display(Image(filename=str(ROOT / grid)))

## 7. Handcrafted Features

The simplest baseline uses classical features:

- HSV color histogram for color distribution
- Canny edge density for edge/structure information
- Laplacian variance for image sharpness

These features are quick to compute and easy to explain.

In [ ]:
def simple_features(image_rgb: np.ndarray) -> np.ndarray:
    """Extract color, edge, and sharpness features from one RGB image."""

    resized = cv2.resize(image_rgb, (160, 160))
    hsv = cv2.cvtColor(resized, cv2.COLOR_RGB2HSV)

    # Color distribution: 8 bins for each HSV channel.
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8], [0, 180, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist).flatten()

    gray = cv2.cvtColor(resized, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 80, 160)
    edge_density = np.array([edges.mean() / 255.0])

    # Higher Laplacian variance usually indicates a sharper image.
    sharpness = np.array([cv2.Laplacian(gray, cv2.CV_64F).var() / 1000.0])

    return np.concatenate([hist, edge_density, sharpness])


def load_image_variant(row: pd.Series, variant: str) -> np.ndarray | None:
    """Load either a raw frame or one enhanced variant."""

    if variant == "raw":
        path = ROOT / row["frame_file"]
    else:
        path = ENHANCED_DIR / variant / row["video_id"] / Path(row["frame_file"]).name

    image_bgr = cv2.imread(str(path))
    if image_bgr is None:
        return None
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def build_feature_table(frame_index: pd.DataFrame, variant: str) -> tuple[np.ndarray, np.ndarray]:
    """Build handcrafted feature matrix X and label vector y for one image variant."""

    features = []
    labels = []

    for _, row in frame_index.iterrows():
        image_rgb = load_image_variant(row, variant)
        if image_rgb is None:
            continue
        features.append(simple_features(image_rgb))
        labels.append(row["video_id"])

    return np.vstack(features), np.array(labels)

## 8. Shared Train/Test Split

All baselines use the same split logic.

The split is stratified by weak video-level label so each class is represented in train and test.

In [ ]:
def split_labels(y: np.ndarray) -> tuple[LabelEncoder, np.ndarray, np.ndarray]:
    """Encode labels and return stratified train/test indices."""

    label_encoder = LabelEncoder()
    y_enc = label_encoder.fit_transform(y)
    indices = np.arange(len(y_enc))

    train_idx, test_idx = train_test_split(
        indices,
        test_size=0.3,
        random_state=SEED,
        stratify=y_enc,
    )

    return label_encoder, train_idx, test_idx

## 9. Tiny CNN Baseline

This CNN is trained from scratch.

It is useful as a baseline, but the dataset is small, so it is expected to be weaker than pretrained ResNet18 embeddings.

In [ ]:
def build_cnn_table(frame_index: pd.DataFrame, variant: str, image_size: int = 96) -> tuple[np.ndarray, np.ndarray]:
    """Load image tensors for Tiny CNN training."""

    images = []
    labels = []

    for _, row in frame_index.iterrows():
        image_rgb = load_image_variant(row, variant)
        if image_rgb is None:
            continue

        resized = cv2.resize(image_rgb, (image_size, image_size)).astype(np.float32) / 255.0
        images.append(np.transpose(resized, (2, 0, 1)))
        labels.append(row["video_id"])

    return np.stack(images), np.array(labels)


def cnn_accuracy(
    frame_index: pd.DataFrame,
    variant: str,
    epochs: int = 12,
    batch_size: int = 16,
) -> dict[str, object]:
    """Train and evaluate the Tiny CNN for one image variant."""

    X, y = build_cnn_table(frame_index, variant)
    label_encoder, train_idx, test_idx = split_labels(y)
    y_enc = label_encoder.transform(y)

    X_train = torch.tensor(X[train_idx], dtype=torch.float32)
    y_train = torch.tensor(y_enc[train_idx], dtype=torch.long)
    X_test = torch.tensor(X[test_idx], dtype=torch.float32)
    y_test = torch.tensor(y_enc[test_idx], dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )

    model = TinyCNN(num_classes=len(label_encoder.classes_))
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    model.train()
    for _ in range(epochs):
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(X_test)
        preds = torch.argmax(logits, dim=1).cpu().numpy()

    return {
        "accuracy": float(accuracy_score(y_enc[test_idx], preds)),
        "report": classification_report(
            y_enc[test_idx],
            preds,
            target_names=label_encoder.classes_,
            zero_division=0,
            output_dict=True,
        ),
        "epochs": epochs,
        "batch_size": batch_size,
        "image_size": int(X.shape[-1]),
    }

## 10. ResNet18 Embeddings

ResNet18 is used as a fixed feature extractor.

The final classification layer is removed, so each frame becomes a compact embedding. Logistic regression and k-NN then classify those embeddings into weak video-level labels.

In [ ]:
def select_torch_device() -> torch.device:
    """Use Apple Silicon MPS when available, otherwise CPU."""

    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def build_resnet18_embeddings(
    frame_index: pd.DataFrame,
    variant: str,
    batch_size: int = 32,
) -> tuple[np.ndarray, np.ndarray, str]:
    """Extract pretrained ResNet18 embeddings for one image variant."""

    device = select_torch_device()
    weights = ResNet18_Weights.DEFAULT
    model = resnet18(weights=weights)

    # Remove the ImageNet classifier so the network outputs feature embeddings.
    model.fc = nn.Identity()
    model.eval()
    model.to(device)

    preprocess = weights.transforms()
    tensors = []
    labels = []

    for _, row in frame_index.iterrows():
        image_rgb = load_image_variant(row, variant)
        if image_rgb is None:
            continue

        image_tensor = torch.from_numpy(image_rgb).permute(2, 0, 1)
        tensors.append(preprocess(image_tensor))
        labels.append(row["video_id"])

    embeddings = []
    with torch.inference_mode():
        for i in range(0, len(tensors), batch_size):
            batch = torch.stack(tensors[i : i + batch_size]).to(device)
            output = model(batch).cpu().numpy()
            embeddings.append(output)

    return np.vstack(embeddings), np.array(labels), str(device)


def run_embedding_baseline(frame_index: pd.DataFrame, variant: str) -> dict[str, object]:
    """Train logistic regression and k-NN on ResNet18 embeddings."""

    X, y, device = build_resnet18_embeddings(frame_index, variant)
    label_encoder, train_idx, test_idx = split_labels(y)
    y_enc = label_encoder.transform(y)

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_enc[train_idx], y_enc[test_idx]

    models = {
        "resnet18_logreg": LogisticRegression(max_iter=2000),
        "resnet18_knn": KNeighborsClassifier(n_neighbors=3),
    }

    metrics: dict[str, object] = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        metrics[name] = {
            "accuracy": float(accuracy_score(y_test, preds)),
            "device": device,
            "report": classification_report(
                y_test,
                preds,
                target_names=label_encoder.classes_,
                zero_division=0,
                output_dict=True,
            ),
        }

    return metrics

## 11. Run All Recognition Baselines

For each variant, the notebook evaluates:

- logistic regression on handcrafted features
- k-NN on handcrafted features
- Tiny CNN
- logistic regression on ResNet18 embeddings
- k-NN on ResNet18 embeddings

In [ ]:
def run_recognition_baseline(frame_index: pd.DataFrame) -> dict[str, dict[str, object]]:
    """Run every recognition baseline for raw and enhanced variants."""

    metrics: dict[str, dict[str, object]] = {}
    variants = ["raw"] + [pipeline.name for pipeline in PIPELINES]

    for variant in variants:
        print(f"Running baselines for: {variant}")

        X, y = build_feature_table(frame_index, variant)
        label_encoder, train_idx, test_idx = split_labels(y)
        y_enc = label_encoder.transform(y)

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y_enc[train_idx], y_enc[test_idx]

        models = {
            "logreg": LogisticRegression(max_iter=2000),
            "knn": KNeighborsClassifier(n_neighbors=3),
        }

        variant_metrics: dict[str, object] = {}
        for name, model in models.items():
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            variant_metrics[name] = {
                "accuracy": float(accuracy_score(y_test, preds)),
                "report": classification_report(
                    y_test,
                    preds,
                    target_names=label_encoder.classes_,
                    zero_division=0,
                    output_dict=True,
                ),
            }

        variant_metrics["tinycnn"] = cnn_accuracy(frame_index, variant)
        variant_metrics.update(run_embedding_baseline(frame_index, variant))
        metrics[variant] = variant_metrics

    return metrics

## 12. Build Results Tables

The metrics dictionary is converted into a compact accuracy table and saved as CSV.

In [ ]:
def build_accuracy_table(metrics: dict[str, dict[str, object]]) -> list[dict[str, object]]:
    """Flatten nested metric dictionaries into sortable accuracy rows."""

    rows = []
    for variant, models in metrics.items():
        for model_name, result in models.items():
            rows.append(
                {
                    "variant": variant,
                    "model": model_name,
                    "accuracy": result["accuracy"],
                    "device": result.get("device", ""),
                }
            )
    return sorted(rows, key=lambda item: item["accuracy"], reverse=True)

In [ ]:
# This cell is the slowest part because it extracts ResNet18 embeddings and trains all baselines.
metrics = run_recognition_baseline(frame_index)
accuracy_rows = build_accuracy_table(metrics)
accuracy_df = pd.DataFrame(accuracy_rows)
accuracy_df.to_csv(ACCURACY_TABLE_PATH, index=False)
accuracy_df.head(12)

## 13. Save Summary Outputs

The summary JSON is useful for the report and presentation because it records the frame count, variants, grids, and full metric dictionary.

In [ ]:
summary = {
    "videos": frame_index["video_id"].nunique(),
    "frames": int(len(frame_index)),
    "pipelines": [pipeline.name for pipeline in PIPELINES],
    "grids": grids,
    "metrics": metrics,
}

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))

print(f"Saved summary to {SUMMARY_PATH.relative_to(ROOT)}")
print(f"Saved accuracy table to {ACCURACY_TABLE_PATH.relative_to(ROOT)}")

## 14. Review Existing Results Without Rerunning

If the full pipeline has already been run, this section reloads the saved outputs and displays the final accuracy table.

In [ ]:
if SUMMARY_PATH.exists() and ACCURACY_TABLE_PATH.exists():
    saved_summary = json.loads(SUMMARY_PATH.read_text())
    saved_accuracy = pd.read_csv(ACCURACY_TABLE_PATH)
    print(saved_summary["videos"], "videos")
    print(saved_summary["frames"], "frames")
    display(saved_accuracy.head(12))
else:
    print("No saved outputs found yet. Run the pipeline cells above first.")